# Part 4 실습 3 — ChatPDF 서비스 구현

## 학습 목표
- PDF 파일을 업로드하고 대화할 수 있는 서비스를 구현한다
- Streamlit으로 RAG 기반 챗봇 UI를 만든다
- 멀티턴 대화(대화 기록 유지)를 RAG와 결합한다

## 이 노트북의 구성
1. **RAG 개념 학습**: RAG가 무엇인지 이해하기
2. **환경 설정**: 필요한 라이브러리 설치 및 API 키 확인
3. **PDF 처리 파이프라인**: PDF 로드 → 청킹 → 벡터 DB
4. **RAG 체인 구축**: 검색 기반 질답 구현
5. **테스트**: 텍스트 파일로 간단히 동작 확인
6. **Streamlit 앱**: 실제 웹 서비스 만들기
7. **배포 및 실행**: 서비스 운영

## 사전 준비 (한 번만 실행하면 됩니다)
터미널에서 다음 명령을 실행하여 필요한 라이브러리를 설치하세요:
```bash
pip install langchain langchain-openai langchain-chroma langchain-community
pip install langchain-text-splitters pypdf streamlit python-dotenv
```

### ChatPDF 서비스의 구조

```
[사용자]
  ↓ PDF 업로드
┌─────────────────────────────┐
│ 1. PDF 로더 (PyPDFLoader)  │ → PDF 파일을 텍스트로 추출
└─────────────────────────────┘
  ↓
┌─────────────────────────────┐
│ 2. 청킹 (Chunking)          │ → 긴 텍스트를 작은 덩어리로 분할
│    (RecursiveCharacter      │   (예: 500자씩)
│     TextSplitter)           │
└─────────────────────────────┘
  ↓
┌─────────────────────────────┐
│ 3. 임베딩 (Embedding)       │ → 각 청크를 벡터로 변환
│    (OpenAI Embeddings)      │   (예: 숫자 배열 1536개)
└─────────────────────────────┘
  ↓
┌─────────────────────────────┐
│ 4. 벡터 DB (Chroma)         │ → 벡터들을 저장하고 검색 가능하게
└─────────────────────────────┘
  ↓
[사용자 질문]
  ↓
┌─────────────────────────────┐
│ 5. 검색 (Retrieval)         │ → 질문과 유사한 청크 찾기 (top-3)
└─────────────────────────────┘
  ↓
┌─────────────────────────────┐
│ 6. 프롬프트 구성            │ → 검색 결과 + 이전 대화 + 질문
└─────────────────────────────┘
  ↓
┌─────────────────────────────┐
│ 7. LLM 생성 (ChatGPT)       │ → 답변 생성
└─────────────────────────────┘
  ↓
[답변 스트리밍]
```

### 핵심 개념들

- **청크 (Chunk)**: PDF를 너무 길면 LLM이 처리하기 어려우므로 작은 조각으로 나눈 것
- **임베딩 (Embedding)**: 텍스트를 숫자 배열로 변환. 의미가 비슷한 텍스트는 벡터도 비슷함
- **벡터 DB**: 벡터들을 빠르게 검색할 수 있는 특수한 데이터베이스
- **멀티턴 (Multi-turn)**: 대화 기록을 기억하면서 여러 번 대화하는 것

  | 개념 | 역할 | 예시 |
  |------|------|------|
  | Document | 텍스트 청크 | PDF의 한 페이지 또는 분할된 텍스트 |
  | Loader | 데이터 소스 읽기 | PyPDFLoader, TextLoader |
  | Splitter | 텍스트 분할 | RecursiveCharacterTextSplitter |
  | Embeddings | 텍스트→벡터 | OpenAIEmbeddings |
  | Vectorstore | 벡터 저장소 | Chroma, Pinecone |
  | Retriever | 문서 검색 | vectorstore.as_retriever() |
  | LLM | 언어 모델 | ChatOpenAI |
  | Prompt | 지시사항 템플릿 | ChatPromptTemplate |
  | Chain | 단계 연결 | LCEL (\| 파이프) |
  | Memory | 대화 기록 | chat_history 리스트 |

## Step 1. 환경 설정

먼저 OpenAI API 키가 설정되었는지 확인합니다.

**주의** .env 파일에서 OPENAI_API_KEY= 형식으로 저장되어 있어야 합니다.

In [5]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

# OPENAI_API_KEY가 설정되어 있는지 확인
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError('❌ OPENAI_API_KEY를 .env 파일에 설정해주세요. (형식: OPENAI_API_KEY=sk-...)')

print('✅ 환경 설정 완료')
print(f'   API 키: {api_key[:20]}...' if api_key else '   API 키: 없음')


✅ 환경 설정 완료
   API 키: sk-proj-JOTZu15pcOwG...


## Step 2. 임베딩(Embedding)과 벡터 DB 개념 학습

### 임베딩이란?

**문제**: 컴퓨터는 텍스트를 직접 비교할 수 없다. 예를 들어 '신발'과 '구두'는 의미가 비슷하지만, 텍스트 자체는 다르다.

**해결책**: 텍스트를 **숫자 배열(벡터)** 로 변환한다.
```
"신발" → [0.1, -0.5, 0.8, 0.2, ..., 0.3]  (1536개 숫자)
"구두" → [0.12, -0.48, 0.81, 0.19, ..., 0.29]
```
**의미**: 의미가 비슷한 단어는 벡터가 비슷하고, 거리(코사인 유사도)가 작다.

### 벡터 DB (Chroma)

**일반 DB VS 벡터DB:** 
- 일반 DB: '신발'을 찾으려면 정확히 '신발'이라고 검색
- 벡터 DB: '구두'로 검색해도 '신발'을 찾을 수 있음 (의미 기반 검색)

**Chroma의 역할:**
1. 청크들의 임베딩을 저장
2. 사용자 질문의 임베딩을 구하기
3. 가장 유사한 청크 top-3 빠르게 찾기



## Step 3. 핵심 함수 - PDF 처리 파이프라인

이 함수는 PDF 파일을 받아서 벡터 DB로 변환합니다. 

구성 요소:
1. PDF 로드: PyPDFLoader로 PDF를 텍스트 페이지로 변환
2. 청킹: RecursiveCharacterTextSplitter로 작은 청크로 분할
3. 임베딩 & 벡터 DB: OpenAI 임베딩 + Chroma 저장

- Loader 구현하기: 로더로 pdf 파일 읽어 페이지 단위로 분리해 저장하기

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('./docs/luckyday.pdf')
pages = loader.load()

print(pages[0].page_content)

C:\Users\qkrru\AppData\Local\Temp\ipykernel_21700\3670317817.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


운수 좋은날
현진건
새침하게 흐린 품이 눈이 올 듯하더니 눈은 아니 오고 얼다가 만 비가 추
적추적 내리는 날이었다.
이날이야말로 동소문 안에서 인력거꾼 노릇을 하는 김첨지에게는 오래간만
에도 닥친 운수 좋은 날이었다 문안에 거기도 문밖은 아니지만 들어간답. ( )
시는 앞집 마마님을 전찻길까지 모셔다 드린 것을 비롯으로 행여나 손님이
있을까 하고 정류장에서 어정어정하며 내리는 사람 하나하나에게 거의 비는
듯한 눈결을 보내고 있다가 마침내 교원인 듯한 양복쟁이를 동광학교(東光
까지 태워다 주기로 되었다) .學校
첫 번에 삼십전 둘째 번에 오십전 아침 댓바람에 그리 흉치 않은 일이, -
었다 그야말로 재수가 옴붙어서 근 열흘 동안 돈 구경도 못한 김첨지는 십.
전짜리 백동화 서 푼 또는 다섯 푼이 찰깍 하고 손바닥에 떨어질 제 거의,
눈물을 흘릴 만큼 기뻤었다 더구나 이날 이때에 이 팔십 전이라는 돈이 그.
에게 얼마나 유용한지 몰랐다 컬컬한 목에 모주 한 잔도 적실 수 있거니와.
그보다도 앓는 아내에게 설렁탕 한 그릇도 사다 줄 수 있음이다.
그의 아내가 기침으로 쿨룩거리기는 벌써 달포가 넘었다 조밥도 굶기를.
먹다시피 하는 형편이니 물론 약 한 첩 써본 일이 없다 구태여 쓰려면 못.
쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자
꾸 온다는 자기의 신조 에 어디까지 충실하였다 따라서 의사에게 보( ) .信條
인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고 일어나기
는 새로 모로도 못 눕는 걸 보면 중증은 중증인 듯 병이 이대도록 심해지.
기는 열흘전에 조밥을 먹고 체한 때문이다 그때도 김첨지가 오래간만에 돈.
을 얻어서 좁쌀 한 되와 십 전짜리 나무 한 단을 사다 주었더니 김첨지의
말에 의지하면 그 오라질 년이 천방지축으로 냄비에 대고 끓였다 마음은.
급하고 불길은 달지 않아 채 익지도 않은 것을 그 오라질년이 숟가락은 고
만두고 손으로 움켜서 두 뺨에 주먹덩이 같은 혹이 불거지도록 누가 빼앗을
듯이 처박질하더니만 그

- Splitter 구현하기: 페이지를 문자 단위로 분리하기

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Loader
loader = PyPDFLoader('./docs/luckyday.pdf')
pages = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    is_separator_regex = False
)
texts = text_splitter.split_documents(pages)

print(texts[0].page_content) # 첫 페이지의 앞부분 500자만 

운수 좋은날
현진건
새침하게 흐린 품이 눈이 올 듯하더니 눈은 아니 오고 얼다가 만 비가 추
적추적 내리는 날이었다.
이날이야말로 동소문 안에서 인력거꾼 노릇을 하는 김첨지에게는 오래간만
에도 닥친 운수 좋은 날이었다 문안에 거기도 문밖은 아니지만 들어간답. ( )
시는 앞집 마마님을 전찻길까지 모셔다 드린 것을 비롯으로 행여나 손님이
있을까 하고 정류장에서 어정어정하며 내리는 사람 하나하나에게 거의 비는
듯한 눈결을 보내고 있다가 마침내 교원인 듯한 양복쟁이를 동광학교(東光
까지 태워다 주기로 되었다) .學校
첫 번에 삼십전 둘째 번에 오십전 아침 댓바람에 그리 흉치 않은 일이, -
었다 그야말로 재수가 옴붙어서 근 열흘 동안 돈 구경도 못한 김첨지는 십.
전짜리 백동화 서 푼 또는 다섯 푼이 찰깍 하고 손바닥에 떨어질 제 거의,
눈물을 흘릴 만큼 기뻤었다 더구나 이날 이때에 이 팔십 전이라는 돈이 그.
에게 얼마나 유용한지 몰랐다 컬컬한 목에 모주 한 잔도 적실 수 있거니와.


- VectorStore 저장하기

In [8]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import os

# PDF 파일 로드
loader = PyPDFLoader('./docs/luckyday.pdf')

# PDF 파일을 페이지별로 분리하여 로드
pages = loader.load()

# 텍스트 분리 설정 (텍스트를 1000자씩 분리, 50자 오버랩)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,  # 각 청크의 크기 (1000자)
    chunk_overlap  = 50,  # 청크 간의 오버랩(50자)
    length_function = len,  # 길이 측정 함수로 len 사용
    is_separator_regex = False,  # 구분자를 정규 표현식으로 사용하지 않음
)

# 문서 페이지를 설정한 기준으로 분리
texts = text_splitter.split_documents(pages)

# OpenAI 임베딩 모델 로드 (텍스트 임베딩을 위해 사용)
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')

# Chroma 데이터베이스가 저장될 디렉토리 경로 설정
persist_directory = './db/chromadb'

# Chroma 데이터베이스가 없으면 새로 생성, 있으면 불러오기
if not os.path.exists(persist_directory):
    # Chroma에 텍스트와 임베딩을 저장
    chromadb = Chroma.from_documents(
        texts,  # 분리된 텍스트
        embeddings_model,  # 임베딩 모델
        collection_name = 'esg',  # 컬렉션 이름
        persist_directory = persist_directory,  # 저장할 디렉토리
    )
else:
    # 기존에 저장된 Chroma 데이터베이스 불러오기
    chromadb = Chroma(
        persist_directory=persist_directory,  # 저장된 디렉토리 경로
        embedding_function=embeddings_model,  # 임베딩 모델
        collection_name='esg'  # 컬렉션 이름
    )

- PDF 경로만 넣으면 벡터 DB 자동으로 만들어주는 함수 정의

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import tempfile
import os

def build_vectorstore_from_pdf(pdf_path: str):
    """
    PDF 파일을 처리해서 벡터 DB를 만듭니다.
    
    Args:
        pdf_path (str): PDF 파일 경로
    
    Returns:
        Chroma: 벡터 스토어 객체
    """
    
    # ─── 1단계: PDF 로드 ───────────────────────────────────────────
    print(f'📄 PDF 파일 처리 중: {pdf_path}')
    
    # PyPDFLoader: PDF 파일을 열어서 페이지별로 분리
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()  # 각 페이지가 하나의 Document 객체
    
    print(f'  ✓ PDF 로드 완료: {len(pages)}개 페이지')
    print(f'    예: 첫 페이지 길이 = {len(pages[0].page_content)} 글자')

    # ─── 2단계: 청킹 (텍스트 분할) ───────────────────────────────────
    # RecursiveCharacterTextSplitter: 텍스트를 작은 청크로 나누기
    # 왜 나누나? LLM은 너무 긴 문맥을 잘 처리하지 못하기 때문
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,      # 한 청크의 크기 (500글자)
        chunk_overlap=50,    # 청크 간 겹침 (검색 정확도 향상)
    )
    chunks = splitter.split_documents(pages)
    
    print(f'  ✓ 청킹 완료: {len(chunks)}개 청크로 분할')
    print(f'    예: 첫 청크 = "{chunks[0].page_content[:100]}..."')

    # ─── 3단계: 임베딩 & 벡터 DB 생성 ──────────────────────────────
    # OpenAIEmbeddings: 각 청크를 벡터(1536개 숫자)로 변환
    # text-embedding-3-small: 빠르고 저렴한 임베딩 모델
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    
    # 임시 디렉토리에 벡터 DB 저장
    persist_dir = tempfile.mkdtemp()
    print(f'  ✓ 임베딩 중... (이 과정은 약간 시간이 걸릴 수 있습니다)')
    
    # Chroma.from_documents: 청크들을 벡터로 변환해서 DB에 저장
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_dir,
    )
    
    count = vectorstore._collection.count()
    print(f'  ✓ 벡터 DB 완료: {count}개 벡터 저장됨')
    print(f'    위치: {persist_dir}')
    
    return vectorstore

print('✅ build_vectorstore_from_pdf 함수 정의 완료')

✅ build_vectorstore_from_pdf 함수 정의 완료


## Step 4. PDF에서 첫 청크 확인하기

청킹이 어떻게 작동하는지 이해하기 위해 실제 테이블을 살펴봄
테스트용 텍스트 파일을 만들어 청크를 확인

In [12]:
from langchain_community.document_loaders import TextLoader

# 테스트용 문서 생성
test_content = """
2026년 인공지능(AI) 기술 동향 보고서

1. 대형 언어 모델(LLM) 시장 동향
2026년 현재 LLM 시장은 OpenAI의 GPT 시리즈, Anthropic의 Claude, Google의 Gemini가 주도하고 있습니다.
특히 멀티모달 기능(텍스트, 이미지, 영상 통합 처리)이 표준화되었습니다.
모든 주요 LLM이 이제 음성 입출력과 이미지 분석을 기본으로 지원합니다.

2. AI 에이전트 기술의 성숙
AI 에이전트는 단순 질답을 넘어 복잡한 작업을 자율적으로 수행합니다.
MCP(Model Context Protocol)가 업계 표준으로 자리잡아 AI와 외부 도구 연결이 표준화되었습니다.
LangGraph, AutoGen, OpenAI Agents SDK 등 다양한 프레임워크가 경쟁 중입니다.

3. 시장 규모와 성장률
글로벌 AI 시장은 2026년 기준 약 5,000억 달러 규모로 성장했습니다.
국내 AI 시장은 약 15조 원으로 전년 대비 40% 성장했습니다.
특히 엔터프라이즈 AI 솔루션과 RAG 기반 챗봇 서비스가 가장 빠르게 성장하고 있습니다.
"""

# 임시 파일로 저장
test_file = './docs/test_report.txt'
with open(test_file, 'w', encoding='utf-8') as f:
    f.write(test_content)

print('✅ 테스트 문서 생성 완료')

✅ 테스트 문서 생성 완료


In [14]:
# 청킹 과정 직접 확인
loader = TextLoader(test_file, encoding='utf-8')
docs = loader.load()
print(f'\n📄 로드된 문서 수: {len(docs)}')
print(f'문서 전체 길이: {len(docs[0].page_content)} 글자')

# 청킹 실행
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 30
)
chunks = splitter.split_documents(docs)

print(f'\n✂️  청킹 완료: {len(chunks)}개 청크')
print(f'\n첫 번째 청크:')
print(f'───────────────────────────────')
print(chunks[0].page_content)
print(f'───────────────────────────────')
print(f'길이: {len(chunks[0].page_content)} 글자')

print(f'\n두 번째 청크:')
print(f'───────────────────────────────')
print(chunks[1].page_content)
print(f'───────────────────────────────')
print(f'길이: {len(chunks[1].page_content)} 글자')
print(f'\n💡 주목: 두 청크가 약간 겹치고 있습니다 (chunk_overlap=30). 이는 검색 정확도를 높입니다.')


📄 로드된 문서 수: 1
문서 전체 길이: 547 글자

✂️  청킹 완료: 3개 청크

첫 번째 청크:
───────────────────────────────
2026년 인공지능(AI) 기술 동향 보고서

1. 대형 언어 모델(LLM) 시장 동향
2026년 현재 LLM 시장은 OpenAI의 GPT 시리즈, Anthropic의 Claude, Google의 Gemini가 주도하고 있습니다.
특히 멀티모달 기능(텍스트, 이미지, 영상 통합 처리)이 표준화되었습니다.
모든 주요 LLM이 이제 음성 입출력과 이미지 분석을 기본으로 지원합니다.
───────────────────────────────
길이: 212 글자

두 번째 청크:
───────────────────────────────
2. AI 에이전트 기술의 성숙
AI 에이전트는 단순 질답을 넘어 복잡한 작업을 자율적으로 수행합니다.
MCP(Model Context Protocol)가 업계 표준으로 자리잡아 AI와 외부 도구 연결이 표준화되었습니다.
LangGraph, AutoGen, OpenAI Agents SDK 등 다양한 프레임워크가 경쟁 중입니다.
───────────────────────────────
길이: 183 글자

💡 주목: 두 청크가 약간 겹치고 있습니다 (chunk_overlap=30). 이는 검색 정확도를 높입니다.


## Step 5. RAG + 멀티턴 대화 체인

이제 벡터 DB를 받아서 RAG 기반 대화형 체인을 만듭니다.

### 동작 원리
1. **검색 (Retrieval)**: 사용자 질문과 유사한 청크 top-3 찾기
2. **맥락 준비**: 검색된 청크 + 이전 대화 + 현재 질문을 프롬프트에 담기
3. **생성 (Generation)**: ChatGPT에게 답변 생성 요청
4. **멀티턴**: 이전 대화 기록을 유지하면서 자연스러운 대화

### 질의 응답 과정 만들기

In [9]:
# 1. 라이브러리 로드
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


# 2. 문서 로드 및 청크 분할
def load_and_split_pdf(pdf_path: str, chunk_size: int = 1000, chunk_overlap: int = 50):
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(pages)

# 3. Chroma 벡터스토어 생성 또는 로드
def create_or_load_chroma(docs, persist_dir="./db/chromadb", collection_name="esg"):
    embeddings = OpenAIEmbeddings()
    if not os.path.exists(persist_dir):
        vectorstore = Chroma.from_documents(
            docs,
            embedding=embeddings,
            collection_name=collection_name,
            persist_directory=persist_dir,
        )
    else:
        vectorstore = Chroma(
            persist_directory=persist_dir,
            embedding_function=embeddings,
            collection_name=collection_name
        )
    return vectorstore

# 4. LCEL 기반 체인 구성: retriever | (prompt | llm | parser)
def build_lcel_chain(vectorstore, model_name="gpt-4o-mini"):
    retriever = vectorstore.as_retriever(search_kwargs={"k": 15})

    prompt = PromptTemplate.from_template(
        """
        너는 문서를 기반으로 질문에 답변하는 유용한 AI야.
        다음 문서를 참고해서 사용자 질문에 답변해줘.

        문서:
        {context}

        질문:
        {question}
        """
    )
    
    # 문서 리스트를 문자열로 변환하는 함수
    format_docs = lambda docs: "\n\n".join(doc.page_content for doc in docs)
    
    llm = ChatOpenAI(model=model_name, temperature=0)
    parser = StrOutputParser()
    
    # LCEL 체인 구성
    chain = (
        RunnableLambda(lambda x: x["question"])  # 문자열 질문만 추출
        | retriever
        | RunnableLambda(lambda docs_and_q: {
            "context": format_docs(docs_and_q),
            "question": question  # 또는 docs_and_q["question"] 
        })
        | prompt
        | llm
        | parser
    )
    return chain

# 5. 실행 함수
def run_chat_lcel(pdf_path, question):
    docs = load_and_split_pdf(pdf_path)
    vectorstore = create_or_load_chroma(docs)
    chain = build_lcel_chain(vectorstore)
    return chain.invoke({"question": question})

pdf_path = "./docs/luckyday.pdf"
question = "아내가 먹고 싶어하는 음식은 무엇이야?"

result = run_chat_lcel(pdf_path, question)

print("\n[질문]")
print(question)
print("\n[답변]")
print(result)


[질문]
아내가 먹고 싶어하는 음식은 무엇이야?

[답변]
아내가 먹고 싶어하는 음식은 설렁탕입니다. 김첨지는 아내가 사흘 전부터 설렁탕 국물이 마시고 싶다고 졸랐다고 언급하고 있습니다.


## Step 6. Streamlit 앱 파일 작성

아래 셀을 실행하면 `chatpdf_app.py` 파일이 생성됩니다.
이것이 실제 웹 서비스 코드입니다.

In [12]:
app_code =r'''
# chatpdf_app.py
# 실행: streamlit run chatpdf_app.py

import os
import tempfile
import streamlit as st
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

# ─── 페이지 설정 ───────────────────────────────────────────────
st.set_page_config(
    page_title="ChatPDF",
    page_icon="📄",
    layout="wide",  # 넓은 레이아웃: 사이드바 + 메인 영역
)
st.title("📄 ChatPDF")
st.caption("PDF 파일을 업로드하고 내용에 대해 자유롭게 질문하세요.")

# ─── PDF 처리 함수 (캐시: 성능 최적화) ─────────────────────────
@st.cache_resource(show_spinner="PDF를 분석하는 중...")
def process_pdf(file_bytes: bytes, filename: str):
    """
    업로드된 PDF를 처리해서 RAG를 위한 벡터 스토어를 반환합니다.
    
    @st.cache_resource: 
      - 같은 파일을 다시 업로드하면 재처리하지 않음
      - 성능 향상: PDF 분석은 시간이 걸리므로 캐싱이 필수
    """
    # 임시 파일로 저장 (Streamlit에 업로드된 파일은 메모리에만 있음)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(file_bytes)
        tmp_path = tmp.name

    # Step 1. PDF 로드
    pages = PyPDFLoader(tmp_path).load()
    
    # Step 2. 청킹 (텍스트 분할)
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=500, 
        chunk_overlap=50
    ).split_documents(pages)

    # Step 3. 임베딩 & 벡터 DB
    persist_dir = tempfile.mkdtemp()
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
        persist_directory=persist_dir,
    )
    
    # 임시 파일 삭제
    os.unlink(tmp_path)
    
    return vectorstore, len(pages), len(chunks)

def build_chain(vectorstore):
    """
    벡터 스토어로부터 RAG + 멀티턴 체인을 구성합니다.
    """
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    # streaming=True: 글자가 하나씩 나오는 효과
    llm = ChatOpenAI(
        model="gpt-4o-mini", 
        temperature=0,
        streaming=True
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "당신은 PDF 문서 전문 분석가입니다.\n"
         "아래 제공된 PDF 문서 내용을 바탕으로만 질문에 정확하게 답하세요.\n"
         "문서에 없는 내용은 '이 문서에서 확인되지 않는 내용입니다'라고 명확히 답하세요.\n\n"
         "[PDF 문서 내용]\n{context}"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ])

    def format_docs(docs):
        return "\n\n".join(
            f"[페이지 {doc.metadata.get('page', '?')+1}] {doc.page_content}"
            for doc in docs
        )

    return (
        {
            "context": (lambda x: x["question"]) | retriever | format_docs,
            "question": lambda x: x["question"],
            "chat_history": lambda x: x.get("chat_history", []),
        }
        | prompt | llm | StrOutputParser()
    )

# ─── 사이드바: PDF 업로드 ──────────────────────────────────────
with st.sidebar:
    st.header("📁 PDF 업로드")
    uploaded = st.file_uploader("PDF 파일을 선택해주세요", type="pdf")

    if uploaded:
        vectorstore, n_pages, n_chunks = process_pdf(
            uploaded.read(), uploaded.name
        )
        st.success(f"✅ PDF 분석 완료!")
        st.info(f"📊 {n_pages}페이지 → {n_chunks}개 청크로 분할됨")

        if st.button("🗑️ 대화 초기화", use_container_width=True):
            st.session_state.messages = []
            st.rerun()
    else:
        st.warning("⚠️ 좌측의 파일 업로더에서 PDF를 선택해주세요.")

# ─── 대화 히스토리 초기화 ──────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []

# ─── 대화 표시 (이전 대화 내역) ────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# ─── 사용자 입력 처리 ──────────────────────────────────────────
if user_input := st.chat_input("PDF 내용에 대해 질문하세요..."):
    if not uploaded:
        st.warning("⚠️ 먼저 PDF를 업로드해주세요.")
        st.stop()

    # 사용자 메시지 표시
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)

    # LangChain 메시지 형식으로 변환 (이전 대화 기록)
    lc_history = []
    for m in st.session_state.messages[:-1]:  # 현재 질문 제외
        if m["role"] == "user":
            lc_history.append(HumanMessage(content=m["content"]))
        else:
            lc_history.append(AIMessage(content=m["content"]))

    # AI 응답 생성 (스트리밍)
    chain = build_chain(vectorstore)
    with st.chat_message("assistant"):
        # st.write_stream: 응답을 스트리밍으로 표시
        # 글자가 하나씩 나타나는 효과
        response = st.write_stream(
            chain.stream({
                "question": user_input,
                "chat_history": lc_history
            })
        )

    # 대화 기록에 저장
    st.session_state.messages.append({"role": "assistant", "content": response})
'''

# chatpdf_app.py 파일로 저장
with open('chatpdf_app.py', 'w', encoding='utf-8') as f:
    f.write(app_code.strip())